# Cross correlation
The cross correlation is a tool for time aligning two signals $x(n)$ and $y(n)$. It is evaluated by

$\varphi_{xy}(m)=\sum_n x(n)\cdot y(n+m)=x(-m)*y(m)$

It can be evaluated in a fast way by the Discrete Fourier Transform (DFT), by

$\varphi_{xy} = \text{DFT}^{-1}\left(\left(\text{DFT}\left(x\right)\right)^* \cdot \text{DFT}\left(y\right)\right)$


In [4]:
import numpy as np

def CrossCorrelation(x, y):
    MaxLength = np.maximum(x.shape[0], y.shape[0])
    FFTLength = 2 ** int(np.ceil(np.log2(2 * MaxLength - 1)))
    X = np.fft.rfft(x, n=FFTLength)
    Y = np.fft.rfft(y, n=FFTLength)
    Phi = np.conj(X) * Y
    phi = np.fft.irfft(Phi)
    return phi

x = np.random.rand(1000)
y = np.random.rand(x.shape[0])
phi = CrossCorrelation(x, y)
for m in range(x.shape[0] // 10):
    assert np.isclose(phi[m], np.sum(x[:x.shape[0]-m] * y[m:]), atol=1e-6), 'wrong evaluation of cross correlation'

phi2 = CrossCorrelation(y, x)
for m in range(x.shape[0] // 10):
    assert np.isclose(phi2[m], phi[-m], atol=1e-6), 'erroy in symmetry'


## Auto correlation
The auto correlation is identical to the cross correlation for $x(n)=y(n)$:

$\varphi_{xx}(m)=\sum_n x(n)\cdot x(n+m)=x(-m)*x(m)$

It has two important properties:

- The unique maximum is at position $m=0$. This maximum corresponds to the [energy](Energy.ipynb) of the signal.
- It is symmetric: $\varphi_{xx}(m)=\varphi_{xx}(-m)$.

In [5]:
def Autocorrelation(x):
    return CrossCorrelation(x, x)

phi3 = Autocorrelation(x)
assert np.argmax(phi3) == 0, 'wrong position of maximum of autocorrelation'

for m in range(x.shape[0] // 10):
    assert np.isclose(phi3[m+1], phi3[-(m+1)], atol=1e-6), 'error in symmetry of autocorrelation'

## Normalised correlation and orthogonality
The cross correlation between two signals $x(n)$ and $y(n)$ can reach any value in the range between $-\sqrt{\sum_n x^2(n) \cdot \sum_n y^2(n)}$ and $\sqrt{\sum_n x^2(n) \cdot \sum_n y^2(n)}$.
In order to compare two different cross correlations directly, the normalised correlation is introduced by the following steps:

1) Evaluate the cross correlation $\varphi(m)$ between two signals $x(n)$ and $y(n)$.

2) Divide $\varphi(m)$  by $\sqrt{\sum_n x^2(n) \cdot \sum_n y^2(n)}$.

The result is called the normalized cross correlation coefficient $\varphi_{\text{norm}}$. The normalized cross correlation coefficient is limited to the range $0\leq\left|\varphi_{\text{norm}}\right|\leq 1$. A value of $\varphi_{\text{norm}}\approx 1$ corresponds to high correlation. In this case, the following is true:

$x(n)\approx c\cdot y(n)$, with $c$ being a real number unequal zero.

A value of $\varphi_{\text{norm}}\approx 0$ means, that both signals $x(n)$ and $y(n)$ are nearly uncorrelated. In this case, both signals are also called orthogonal.

In [ ]:
def NormalizedCrossCorrelation(x, y):
    phi = CrossCorrelation(x, y)
    return phi / np.sqrt(np.sum(x**2) * np.sum(y**2))

phi4 = NormalizedCrossCorrelation(x, y)
assert np.amax(phi4) <= 1.0, 'normalized cross correlation should be less than or equal to 1'
assert np.amin(phi4) >= -1.0, 'normalized cross correlation should be greater than or equal to -1'
print('All tests passed successfully')

All tests passed successfully
